# AsyncFlow — MMc Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **multi-server** scenario compatible with **M/M/c** assumptions
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)




In [111]:
import sys, importlib


for m in list(sys.modules):
    if m.startswith("asyncflow"):
        del sys.modules[m]


from asyncflow import AsyncFlow, SimulationRunner
from asyncflow.analysis import MMc, ResultsAnalyzer
from asyncflow.components import (
    Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
)
from asyncflow.settings import SimulationSettings

import simpy

In [112]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
from asyncflow.settings import SimulationSettings
from asyncflow.analysis import  ResultsAnalyzer, SweepAnalyzer, MMc
from asyncflow.enums import Distribution

print("Imports OK.")

Imports OK.


## 1) Build an M/M/c split-friendly scenario

* **Multiple identical servers with exponential CPU service**
  Topology includes **\$c \geq 2\$ identical servers**, each exposing exactly **one endpoint** with exactly **one CPU-bound step**.
  Service times follow an **Exponential** distribution with mean \$E\[S]\$ (service rate \$\mu = 1/E\[S]\$). No RAM/IO steps are included in the pipeline.

* **Load balancer with FCFS dispatch**

* **“Poisson arrivals” via the generator**
  
  

---

```mermaid
graph LR;
    rqs1["<b>RqsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    lb1["<b>LoadBalancer</b><br/>id: lb-1<br/>Policy: round_robin"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]
    app2["<b>Server</b><br/>id: app-2<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-lb<br/>Latency: 0.0001" --> lb1;
    lb1 -- "Dispatch<br/>Edge: lb-app1<br/>Latency: 0.0001" --> app1;
    lb1 -- "Dispatch<br/>Edge: lb-app2<br/>Latency: 0.0001" --> app2;
    app1 -- "Response<br/>Edge: app1-client<br/>Latency: 0.0001" --> client1;
    app2 -- "Response<br/>Edge: app2-client<br/>Latency: 0.0001" --> client1;
```

---



In [113]:
def build_payload():
    generator = ArrivalsGenerator(
        id="rqs-1",
        lambda_rps=270,
        model=Distribution.POISSON
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",
                "step_operation": {
                    "cpu_time": {"mean": 0.01, "distribution": "exponential"},
                },
            },
        ],
    )

    srv1 = Server(
        id="srv-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    srv2 = Server(
        id="srv-2",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    
    srv3 = Server(
        id="srv-3",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    lb = LoadBalancer(
        id="lb-1",
        algorithms="fcfs",  
        server_covered={"srv-1", "srv-2", "srv-3"},
    )

    edges = [
        LinkEdge(id="gen-client",  source="rqs-1",  target="client-1",),
        LinkEdge(id="client-lb",   source="client-1", target="lb-1",  ),
        LinkEdge(id="lb-srv1",     source="lb-1",   target="srv-1",   ),
        LinkEdge(id="lb-srv2",     source="lb-1",   target="srv-2",   ),
        LinkEdge(id="lb-srv3",     source="lb-1",   target="srv-3",   ),
        LinkEdge(id="srv1-client", source="srv-1",  target="client-1",),
        LinkEdge(id="srv2-client", source="srv-2",  target="client-1",),
        LinkEdge(id="srv3-client", source="srv-3",  target="client-1",),
    ]

    settings = SimulationSettings(
        total_simulation_time=3600,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_arrivals_generator(generator)
        .add_client(client)
        .add_servers(srv1, srv2, srv3)
        .add_load_balancer(lb)
        .add_edges(*edges)
        .add_simulation_settings(settings)
    ).build_payload()

    return payload


## 2) Run the simulation

In [114]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")


Done.


# 3) M/M/c (FCFS) — theory vs observed comparison

This section shows how we compute the **theoretical Erlang-C KPIs** (pooled queue, FCFS) and compare them against **simulation estimates**.

---

## Variables

* **$c$**: number of identical servers.
* **$\lambda$**: global arrival rate (req/s).
* **$\mu$**: per-server service rate (req/s), $\mu = 1/\mathbb{E}[S]$.
* **$\rho$**: global utilization, $\rho = \lambda/(c\mu)$.
* **$W$**: mean time in system (queue + service).
* **$W_q$**: mean waiting time in queue.
* **$L$**: mean number in system.
* **$L_q$**: mean number in queue.

---

## Theory (Erlang-C formulas)

We assume **Poisson arrivals** for $\lambda$ (taken directly from the payload).

1. Offered load:

$$
a = \frac{\lambda}{\mu}
$$

2. Probability system is empty:

$$
P_0 = \left[\sum_{n=0}^{c-1}\frac{a^n}{n!} + \frac{a^c}{c!\,(1-\rho)}\right]^{-1}
$$

3. Probability of waiting (Erlang-C):

$$
P_w = \frac{a^c}{c!\,(1-\rho)} \, P_0
$$

4. Queue length and waiting:

$$
L_q = P_w \cdot \frac{\rho}{1-\rho}, \qquad
W_q = \frac{L_q}{\lambda}
$$

5. Total response time and system size:

$$
W = W_q + \frac{1}{\mu}, \qquad
L = \lambda W
$$

If $\rho \ge 1$, the system is unstable and all metrics diverge to $+\infty$.

---

## Observed (from simulation)

After processing metrics:

1. **Arrival rate**:

$$
\lambda_{\text{Observed}} = \text{mean throughput (client completions)}
$$

2. **Service rate**:

$$
\mu_{\text{Observed}} = 1 / \overline{S}, \quad \overline{S} = \text{mean(service\_time)}
$$

3. **End-to-end latency**:

$$
W_{\text{Observed}} = \text{mean(client latencies)}
$$

4. **Waiting time**:

$$
W_{q,\text{Observed}} = \text{mean(waiting\_time)} 
$$

5. **Little’s law check**:

$$
L_{\text{Observed}} = \lambda_{\text{Observed}} W_{\text{Observed}}, \qquad
L_{q,\text{Observed}} = \lambda_{\text{Observed}} W_{q,\text{Observed}}
$$

6. **Utilization**:

$$
\rho_{\text{Observed}} = \lambda_{\text{Observed}}/(c\,\mu_{\text{Observed}})
$$

---

## Comparison

The analyzer builds a table with two columns — **Theory** (Erlang-C closed forms) and **Observed** (empirical estimates) — and reports absolute and relative deltas.

This allows us to verify whether AsyncFlow reproduces the textbook M/M/c (FCFS) predictions under Poisson arrivals and exponential service.




In [115]:
mmc = MMc()
if mmc.is_compatible(payload):
   mmc.print_comparison(payload, results)  
else:
    print("Payload is not compatible with M/M/c:")
    for reason in mmc.explain_incompatibilities(payload):
        print(" -", reason)
   


MMc (FCFS/Erlang-C) — Theory vs Observed
-------------------------------------------------------------------
sym  metric                    theory    observed        abs   rel%
-------------------------------------------------------------------
λ    Arrival rate (1/s)    270.000000  269.997500  -0.002500  -0.00
μ    Service rate (1/s)    100.000000   99.925514  -0.074486  -0.07
rho  Utilization             0.900000    0.900663   0.000663   0.07
L    Mean items in sys      10.053549    9.991820  -0.061730  -0.61
Lq   Mean items in queue     7.353549    7.289848  -0.063701  -0.87
W    Mean time in sys (s)    0.037235    0.037007  -0.000228  -0.61
Wq   Mean waiting (s)        0.027235    0.027000  -0.000236  -0.87
